In [9]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import time
import glob
import warnings
warnings.filterwarnings('ignore')


#data
path = "E:\\daily_data"
all_files = glob.glob(path + "/*.csv.gz")   #把所有的数据文件文件名读取在一起

li = []


for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0, compression="gzip")
    df["date"] = filename[14:-7]
    li.append(df)


frame = pd.concat(li, axis=0, ignore_index=True)
result_df = frame.sort_values(by=['ticker', 'date'], ascending=True)
result_df = result_df.reset_index(drop=True)
result_df.dropna(inplace=True)

In [14]:
df = result_df.copy()
df["avg_price"] = ((df["open"]+df["close"]+df["low"])/3)
# df['upshadow'] = (df['high'] - (df[['close', 'open']].max(axis=1)))
# df['downshadow'] = ((df[['close', 'open']].min(axis=1)) - df['low'])

In [15]:
df

,ticker,volume,open,close,high,low,window_start,transactions,date,avg_price
0,A,1392130,85.9000,85.9500,86.3500,85.2000,1577941200000000000,15683,2020-01-02,85.683333
1,A,1117939,84.6700,84.5700,85.3300,84.5000,1578027600000000000,13063,2020-01-03,84.580000
2,A,1993195,84.0000,84.8200,84.8200,83.6000,1578286800000000000,18253,2020-01-06,84.140000
3,A,1723146,83.9600,85.0800,85.2600,83.9400,1578373200000000000,17798,2020-01-07,84.326667
4,A,1842530,85.9600,85.9200,86.4700,85.2000,1578459600000000000,20009,2020-01-08,85.693333
...,...,...,...,...,...,...,...,...,...,...
14113260,ZZZ,351,27.6100,27.6100,27.6100,27.6100,1748836800000000000,28,2025-06-02,27.610000
14113261,ZZZ,754,27.8200,27.9647,27.9647,27.7800,1748923200000000000,35,2025-06-03,27.854900
14113262,ZZZ,1028,27.9300,27.8829,27.9597,27.8829,1749009600000000000,24,2025-06-04,27.898600
14113263,ZZZ,111,27.5444,27.5444,27.5444,27.5444,1749096000000000000,12,2025-06-05,27.544400


In [11]:
#将日期列转换为datetime类型
df['date']=pd.to_datetime(df['date'])

#保存文件夹路径
output_folder='E:\\new_feature\\updown2'
selected_columns=['ticker', "avg_price"]

os.makedirs(output_folder,exist_ok=True)

#创建日期索引并检查是否在dateframe中存在
date_index=pd.date_range(df['date'].min(),df['date'].max(),freq='D')
existing_dates=[d for d in date_index if d in df['date'].values]

#遍历每个日期如果它在dataframe中存在 则将其保存为单独的csv文件
for date in existing_dates:
    group=df.loc[df['date']==date,selected_columns]
    filename=os.path.join(output_folder,f'{date.strftime("%Y-%m-%d")}.csv')
    group=group.sort_values(by='ticker',ascending=True)
    group.to_csv(filename,index=False)

In [7]:
#标准化上下影线
df['up'] = df['upshadow'] / df.groupby('ticker')['upshadow'].transform(lambda x: x.rolling(window=5).mean())
df['down'] = df['downshadow'] / df.groupby('ticker')['downshadow'].transform(lambda x: x.rolling(window=5).mean())

In [9]:
#选股因子
df['up_mean'] = df.groupby('ticker')['up'].transform(lambda x: x.rolling(window=20).mean())
df['up_std'] = df.groupby('ticker')['up'].transform(lambda x: x.rolling(window=20).std())
df['down_mean'] = df.groupby('ticker')['down'].transform(lambda x: x.rolling(window=20).mean())
df['down_std'] = df.groupby('ticker')['down'].transform(lambda x: x.rolling(window=20).std())